# 🧠 Multi-Head Attention
**Course:** Deep Learning  
**Lecture:** 06 — Attention Mechanisms and Transformers  
**Part:** b — Multi-Head Attention

---

> **Where we are:** We know how single attention works. Now, we expand it so the model can focus on different aspects of a sequence simultaneously.

**What you'll learn:**
- Multi-Head Attention equations
- Implementing heads in parallel
- Projection matrices $W^Q, W^K, W^V, W^O$

**Exam relevance:** ⭐⭐⭐⭐ (Parameter counting for MHA is a classic exam question)


In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import seaborn as sns

# Confirm device
print(f"TensorFlow {tf.__version__} | Devices: {[d.name for d in tf.config.list_physical_devices()]}")

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12


## 📖 Theory: Why Multiple Heads?

$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h) W^O$

where $\text{head}_i = \text{Attention}(Q W_i^Q, K W_i^K, V W_i^V)$

**Why multiple heads?**
- A single attention head aggregates all information based on one similarity metric.
- With multiple heads, Head 1 might learn syntactic dependencies (e.g., verbs to subjects), while Head 2 learns semantic dependencies (e.g., pronouns to entities).
- The dimension $d_{model}$ is typically divided by $h$ to get $d_k$ (e.g. 512 / 8 = 64). Thus, the computation cost is roughly the same as one large attention head.


## 🔀 Your Options

At this step, you can choose between:
- **Sequential Heads**: Iterate through heads (slow).
- **Parallel Heads (Reshape)**: Treat heads as a dimension in the tensor and compute all at once via matrix multiplication (fast, used in practice).


In [ ]:
# ✏️ Your Turn: Multi-Head Attention Layer

class CustomMultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads):
        super().__init__()
        # ### YOUR CODE HERE ###
        # Initialize num_heads, d_model, and d_k
        # Initialize Dense layers for Wq, Wk, Wv, Wo
        pass
        
    def call(self, x):
        # ### YOUR CODE HERE ###
        # 1. Pass x through Wq, Wk, Wv
        # 2. Reshape to split heads: (batch_size, seq_len, num_heads, d_k)
        # 3. Transpose to (batch_size, num_heads, seq_len, d_k)
        # 4. Compute scaled dot product attention
        # 5. Transpose back and concat
        # 6. Pass through Wo
        pass


In [ ]:
# ✅ Solution: Multi-Head Attention Layer

class CustomMultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        assert d_model % self.num_heads == 0
        self.d_k = d_model // self.num_heads
        
        self.wq = tf.keras.layers.Dense(d_model)
        self.wk = tf.keras.layers.Dense(d_model)
        self.wv = tf.keras.layers.Dense(d_model)
        self.dense = tf.keras.layers.Dense(d_model) # Wo
        
    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.d_k))
        return tf.transpose(x, perm=[0, 2, 1, 3])
        
    def call(self, v, k, q, mask=None):
        batch_size = tf.shape(q)[0]
        
        q = self.wq(q)
        k = self.wk(k)
        v = self.wv(v)
        
        q = self.split_heads(q, batch_size)
        k = self.split_heads(k, batch_size)
        v = self.split_heads(v, batch_size)
        
        matmul_qk = tf.matmul(q, k, transpose_b=True)
        dk_float = tf.cast(self.d_k, tf.float32)
        scaled_attention_logits = matmul_qk / tf.math.sqrt(dk_float)
        
        if mask is not None:
            scaled_attention_logits += (mask * -1e9)
            
        attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
        output = tf.matmul(attention_weights, v)
        
        output = tf.transpose(output, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(output, (batch_size, -1, self.d_model))
        
        return self.dense(concat_attention), attention_weights

sample_mha = CustomMultiHeadAttention(d_model=512, num_heads=8)
dummy_x = tf.random.normal((1, 10, 512)) # (batch, seq_len, d_model)
out, weights = sample_mha(dummy_x, dummy_x, dummy_x)
print("Output shape:", out.shape)
print("Weights shape:", weights.shape)


In [ ]:
# 🔬 Comparison: Visualize Attention Heads

# Let's generate a random set of weights simulating what different heads might look like
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for i in range(4):
    ax = axes[i//2, i%2]
    # In a real model, this would be weights[0, i]
    # Here we show dummy patterns: Head 0 might look at next word, Head 1 at previous, etc.
    sns.heatmap(tf.random.uniform((10, 10)), ax=ax, cmap='Blues', cbar=False)
    ax.set_title(f"Head {i+1}")
plt.tight_layout()
plt.show()

# Parameter count comparison:
# Wq, Wk, Wv, Wo all take (d_model, d_model) parameters
print(f"Total params for d_model=512: {4 * 512 * 512}")


## 🎓 Exam Corner

### Question 1 (Conceptual)
If `d_model=512` and `num_heads=8`, what is `d_k` for each head? What is the total parameter count of a MultiHeadAttention layer?

<details><summary>💡 Answer</summary>

- `d_k` = 512 / 8 = 64.
- Total parameters = 4 weight matrices ($W^Q, W^K, W^V, W^O$).
  Each is `d_model` x `d_model` (plus biases, usually).
  Roughly $4 \times 512^2 = 1,048,576$ parameters (excluding biases).

</details>

### Question 2 (Practical)
Why is Multi-Head Attention better than a single large attention? What is the $W^O$ matrix for and what is its shape?

<details><summary>💡 Answer</summary>

- Multiple heads allow the model to jointly attend to information from different representation subspaces at different positions. A single head averages this out.
- $W^O$ (Output weight matrix) mixes the concatenated outputs from all the different heads back into the original $d_{model}$ dimension. Its shape is `(d_model, d_model)`.

</details>
